# Saving, Loading and Large Data

This example covers:

1. Saving a fitted model to a `.stanbktmod` archive and loading it back.
2. `low_memory=True`, which keeps fits on disk.
3. Memory efficient posterior prediction with the `_summary` methods.

See the [Saving and Loading](../guide/save_load.rst) and [Memory and Performance](../guide/performance.rst) guide pages for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

In [2]:
from stanbkt.fits import MCMCFitOptions
from stanbkt.models import StandardBKT
from stanbkt.utils import VerbosityLevel, sim_simple_BKT

data_df = sim_simple_BKT(n_students=150, n_problems=30, n_kcs=4, rng_seed=51,
                         learn=[0.05, 0.1, 0.15, 0.2], guess=0.2, slip=0.1)
fit_opts = MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False)

model = StandardBKT(verbose=VerbosityLevel.WARN)
model.fit(data_df, stan_fit_options=fit_opts)

StandardBKT(fit_method=<FitMethod.MCMC: 'mcmc'>, verbose=<VerbosityLevel.WARN: 1>, is_fitted=True)

## Save and load

`save` writes the fits, the cached summaries and the model metadata (class, constructor arguments, group and student maps) to a single archive. The `.stanbktmod` suffix is added if missing. `load_model` rebuilds the same model.

In [3]:
import os
import tempfile

from stanbkt import load_model

save_dir = tempfile.mkdtemp()
model.summary()  # cached and saved with the model
model.save(os.path.join(save_dir, "four_kcs"))
print(os.listdir(save_dir))

loaded = load_model(os.path.join(save_dir, "four_kcs"))
type(loaded).__name__, sorted(loaded.fits.get_fitted_kcs())

['four_kcs.stanbktmod']


('StandardBKT', ['kc_0', 'kc_1', 'kc_2', 'kc_3'])

The loaded model predicts exactly like the original:

In [4]:
original = model.predict(data_df)
reloaded = loaded.predict(data_df)
np.allclose(original["pKnow"], reloaded["pKnow"])

True

## Low memory mode

With many KCs, keeping every fit in memory adds up. `low_memory=True` evicts each KC's fit to a temporary directory after fitting and reloads it only when a method needs it. `summary()` and the `predict*` methods release the fits they reloaded when they finish, cached summaries never reload a fit, and `save()` copies evicted fits from disk. A model saved with `low_memory=True` is also loaded lazily.

In [5]:
low_mem = StandardBKT(low_memory=True, verbose=VerbosityLevel.WARN)
low_mem.fit(data_df, stan_fit_options=fit_opts)
low_mem.predict(data_df).head()

,kc_id,student_id,problem_id,pKnow,pCorrectness,correct
0,kc_1,stu_0,prob_0,0.064292,0.258669,0
1,kc_1,stu_0,prob_3,0.112254,0.290587,1
2,kc_1,stu_0,prob_4,0.397837,0.480644,1
3,kc_1,stu_0,prob_5,0.734336,0.704584,1
4,kc_1,stu_0,prob_6,0.897858,0.813408,0


## Posterior prediction on large data

Draw level output has one row per posterior draw and interaction. Here that is 2,000 draws for each interaction:

In [6]:
draws = model.predict_posterior_draws(data_df, backend="numba")
n_rows = sum(len(df) for df in draws.values())
draw_mb = sum(df.memory_usage(deep=True).sum() for df in draws.values()) / 1e6
print(f"{len(data_df):,} interactions -> {n_rows:,} draw rows, {draw_mb:.0f} MB")

4,500 interactions -> 9,000,000 draw rows, 216 MB


The `_summary` methods compute per interaction statistics without keeping all draws, so their output is the size of the data. `backend="numba"` avoids writing Stan CSV output, and `n_cores` runs KCs concurrently at the cost of holding several KCs' draws in memory at once.

In [7]:
del draws

summary = model.predict_posterior_summary(data_df, quantiles=[0.05, 0.95], backend="numba", n_cores=2)
summary_mb = summary.memory_usage(deep=True).sum() / 1e6
print(f"summary: {len(summary):,} rows, {summary_mb:.1f} MB")
summary.head()

summary: 4,500 rows, 0.9 MB


,kc_id,student_id,problem_id,correct,pKnow_mean,pKnow_std,pKnow_median,pKnow_5.00%,pKnow_95.00%,pCorrectness_mean,pCorrectness_std,pCorrectness_median,pCorrectness_5.00%,pCorrectness_95.00%
0,kc_1,stu_0,prob_0,0,0.064292,0.038099,0.062925,0.003029,0.130063,0.258775,0.024619,0.257838,0.219617,0.302570
1,kc_1,stu_0,prob_3,1,0.112627,0.013628,0.111981,0.090933,0.135728,0.290868,0.016675,0.290794,0.264296,0.318102
2,kc_1,stu_0,prob_4,1,0.398313,0.040958,0.398150,0.332578,0.466017,0.481322,0.022165,0.481139,0.445499,0.517956
3,kc_1,stu_0,prob_5,1,0.731972,0.040650,0.734572,0.663066,0.795115,0.703411,0.025279,0.704451,0.660599,0.744793
4,kc_1,stu_0,prob_6,0,0.895643,0.022743,0.897103,0.855808,0.930957,0.811983,0.019647,0.812550,0.779126,0.843880


When parameter uncertainty is not needed, `predict` with point estimates is the cheapest option of all.

In [8]:
import shutil

shutil.rmtree(save_dir)